# One bounded C/D validation comparison
Run these four cells in order, or choose **Runtime > Run all**. Select an **L4 GPU** if available; T4 also works.

This compares the original decision prompt with one revision requiring explicit override justification. Qwen remains the classifier; C/D evidence, model and token budgets are matched. The held-out test is not used.

52 validation cases: 40 label-blind site-balanced windows plus 12 supplemental fault cases. The report separates these cohorts. This is development after the test audit, not a new independent final test.

There are **208 decision calls**. Only if BOTH C and D meet the fixed improvement criteria will the notebook run up to **208 explanation calls**. Otherwise it stops automatically and saves the negative result. No retries or further prompt variants are run. Resume an interruption by rerunning these same cells; cached attempts are retained.


## 1. Connect Drive and verify the synced files

In [ ]:
from pathlib import Path
import os, sys, json, hashlib, subprocess
from google.colab import drive
ROOT = Path('/content/drive/Othercomputers/My Laptop (1)/CP5403')
if not ROOT.is_dir() and not Path('/content/drive/MyDrive').is_dir():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Drive connection failed. Reconnect the runtime and complete Google authorization, then rerun. Do not mount the project folder.') from exc
if not ROOT.is_dir():
    raise FileNotFoundError(f'Drive project not found: {ROOT}. Check Drive desktop sync and the Colab Files panel. Do not force-remount.')
os.chdir(ROOT)
BUNDLE = ROOT / 'Datasets/PHM15-diagnosis/recovery_validation_v1'
sha = lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
assert sha(BUNDLE/'export_contract.json') == 'b0721903a0b30c6e48910b4adc337fc0583db8d8b2e886f93d999c4423134bd2', 'Recovery export not fully synced.'
contract = json.loads((BUNDLE/'export_contract.json').read_text(encoding='utf-8'))
for name, expected in contract['code_sha256'].items():
    assert sha(ROOT/'Files'/name) == expected, f'File not fully synced: {name}'
for name, expected in contract['files'].items():
    assert sha(BUNDLE/name) == expected, f'Bundle file not fully synced: {name}'
print('Recovery files verified. No held-out test data loaded.')
print('Maximum calls:', contract['maximum_calls'])


Mounted at /content/drive
Recovery files verified. No held-out test data loaded.
Maximum calls: {'decisions': 208, 'explanations_if_gate_passes': 208}


## 2. Install the pinned dependencies

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime and rerun.'
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'Files/requirements-literature-colab.txt',
                '-r', 'Files/requirements-qwen-colab.txt', 'onnxruntime==1.31.1', 'lm-format-enforcer==0.11.3'], check=True)
print('Dependencies ready. No Hugging Face approval is required for this Qwen checkpoint.')


GPU: NVIDIA L4
Dependencies ready. No Hugging Face approval is required for this Qwen checkpoint.


## 3. Run checks and prepare the comparison

In [ ]:
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'Files', '-p', 'test_phm_recovery_validation.py', '-v'], check=True)
subprocess.run([sys.executable, 'Files/phm_recovery_validation.py', '--root', str(ROOT), '--prepare-only'], check=True)
print('Ready. The next cell runs the comparison and stops automatically if the improvement gate fails.')


Ready. The next cell runs the comparison and stops automatically if the improvement gate fails.


## 4. Run once and save the result

In [ ]:
 subprocess.run([sys.executable, '-u', 'Files/phm_recovery_validation.py', '--root', str(ROOT)], check=True)
latest = json.loads((ROOT/'Outputs/LATEST_RUN.json').read_text(encoding='utf-8'))
folder = ROOT/latest['run_directory']
print('Latest report:', latest)
print('Improvement gate:', json.loads((folder/'improvement_gate.json').read_text(encoding='utf-8')))
metrics = json.loads((folder/'comparison_metrics.json').read_text(encoding='utf-8'))
for variant in ('original','revised'):
    for condition in ('C','D'):
        m=metrics[variant][condition]
        print(variant,condition,'combined macro-F1:',round(m['all_selected']['macro_f1'],4),
              'natural accuracy:',round(m['natural']['accuracy'],4),
              'harmful overrides:',m['harmful_overrides'])
print('Saved metrics:',folder/'comparison_metrics.json')
print('Let Drive sync, then send this report. Do not rerun failed outputs or change prompts.')


Latest report: {'run_id': '20261122T194335_bounded_CD_override_validation_e6341847', 'run_directory': 'Outputs/runs/20261122T194335_bounded_CD_override_validation_e6341847', 'status': 'completed_with_issues', 'updated_utc': '2026-11-22T21:54:28.663913+00:00'}
Improvement gate: {'passed': False, 'checks': {'C': {'macro_f1_gain_at_least_002': False, 'natural_accuracy_not_lower': False, 'harmful_overrides_reduced': True, 'at_least_95_percent_valid': False, 'neither_fault_recall_lower': False}, 'D': {'macro_f1_gain_at_least_002': False, 'natural_accuracy_not_lower': False, 'harmful_overrides_reduced': True, 'at_least_95_percent_valid': False, 'neither_fault_recall_lower': False}}, 'meaning': 'Pragmatic development stop rule, not statistical significance or independent test confirmation.'}
original C combined macro-F1: 0.4498 natural accuracy: 0.55 harmful overrides: 8
original D combined macro-F1: 0.4426 natural accuracy: 0.525 harmful overrides: 9
revised C combined macro-F1: 0.0 natural 